# Dimensionality Reduction: t-SNE (t-Distributed Stochastic Neighbor Embedding)

This notebook explores t-SNE, a powerful non-linear dimensionality reduction technique optimized for data visualization. We will cover:
1. **PCA vs. t-SNE:** Linear vs. Non-linear projections.
2. **The Perplexity Parameter:** How tuning the "neighborhood size" changes the entire visualization.
3. **The Feature Engineering Trap:** Proving mathematically why t-SNE coordinates cannot be used to train predictive models.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.datasets import make_classification

sns.set_theme(style="whitegrid")

### 1. Generating Complex, High-Dimensional Data
We will generate a dataset with 40 dimensions and 4 distinct classes. We will intentionally add redundant and noisy features to make the "true" clusters difficult to separate using simple linear methods.

In [ ]:
np.random.seed(42)

# 40-Dimensional data with 4 classes
X, y = make_classification(
    n_samples=800, 
    n_features=40, 
    n_informative=15, 
    n_redundant=10,
    n_classes=4, 
    n_clusters_per_class=1, 
    random_state=42
)

print(f"High-Dimensional Space: {X.shape} (800 rows, 40 dimensions)")

### 2. The Showdown: PCA (Global) vs. t-SNE (Local)
PCA attempts to preserve the *global variance* using straight lines. 
t-SNE uses probabilities to preserve *local neighborhoods* (points that are close in 40D should stay close in 2D) and uses the Student's t-distribution to push apart dissimilar clusters (solving the "Crowding Problem").

In [ ]:
# 1. Linear Projection (PCA)
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

# 2. Non-Linear Embedding (t-SNE)
tsne = TSNE(n_components=2, perplexity=30, random_state=42)
X_tsne = tsne.fit_transform(X)

# Visualization Showdown
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=y, palette='Set1', ax=axes[0], s=50, alpha=0.8)
axes[0].set_title("PCA: Global Variance (Classes bleed together)", fontsize=14)

sns.scatterplot(x=X_tsne[:, 0], y=X_tsne[:, 1], hue=y, palette='Set1', ax=axes[1], s=50, alpha=0.8)
axes[1].set_title("t-SNE: Local Neighborhoods (Beautiful Separation)", fontsize=14)

plt.tight_layout()
plt.show()

print("Observation: PCA fails to untangle the 40D clusters because it is restricted to linear transformations. t-SNE dynamically bends and maps the topology, revealing that the 4 classes are actually highly distinct!")

### 3. Tuning Perplexity
`perplexity` is the most critical hyperparameter in t-SNE. It loosely equates to the number of "nearest neighbors" the algorithm considers when calculating probabilities.
* **Low Perplexity (e.g., 5):** Focuses too much on immediate neighbors, creating fragmented, fake micro-clusters.
* **Optimal Perplexity (e.g., 30-50):** Balances local and global structure beautifully.
* **High Perplexity (e.g., 100+):** Considers too many points as neighbors, turning the visualization into a blended blob.

In [ ]:
perplexities = [5, 30, 100]
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, p in enumerate(perplexities):
    # Run t-SNE with different perplexity values
    tsne_p = TSNE(n_components=2, perplexity=p, random_state=42)
    X_p = tsne_p.fit_transform(X)
    
    sns.scatterplot(x=X_p[:, 0], y=X_p[:, 1], hue=y, palette='Set1', ax=axes[i], legend=False, s=40, alpha=0.7)
    axes[i].set_title(f"Perplexity = {p}", fontsize=14)

plt.tight_layout()
plt.show()

### 4. The Engineering Trap: Why t-SNE is NOT for Feature Extraction
You **cannot** use t-SNE to engineer features for training a predictive model. 

Unlike PCA (which is deterministic and finds the exact same axes every time), t-SNE is **Stochastic** (randomized). Let's run t-SNE twice on the exact same data with different random states. If the coordinates change entirely, it proves the features are unstable and mathematically meaningless for a machine learning model.

In [ ]:
# Run 1 (Random State 10)
tsne_run1 = TSNE(n_components=2, perplexity=30, random_state=10).fit_transform(X)

# Run 2 (Random State 99)
tsne_run2 = TSNE(n_components=2, perplexity=30, random_state=99).fit_transform(X)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(x=tsne_run1[:, 0], y=tsne_run1[:, 1], hue=y, palette='Set1', ax=axes[0], legend=False)
axes[0].set_title("t-SNE Output (Run 1)")

sns.scatterplot(x=tsne_run2[:, 0], y=tsne_run2[:, 1], hue=y, palette='Set1', ax=axes[1], legend=False)
axes[1].set_title("t-SNE Output (Run 2)")

plt.show()

print("--- CRITICAL TAKEAWAY ---")
print("Look at the X and Y axes. The clusters have completely rotated and swapped coordinates!")
print("If you trained a Logistic Regression model on Run 1, it would utterly fail on data transformed by Run 2.")
print("Use t-SNE for your EYES (Visualization & EDA). Use PCA/SVD for your MODELS (Feature Engineering).")